# Statistics of Li et al. 2024 (Nat. Mach. Intell.) on our runs + cross-scale statistics
Two groups, each statistic tagged with a flag in its panel title:
- **Fig A `[LI]`** — physical statistics of the paper, single component (`codes/paper_figures_li.py`): increment PDFs (Fig 1a), acceleration PDF and correlation (Fig 2), $S_p$ (Fig 4a, + $p=8$), $F^{(p)}$ (Fig 4b), $\zeta(p,\tau)$ (Fig 5), $E$, $A$ (Table 2). Li et al. call them "multiscale" because they are scanned over the lag $\tau$, but each is a function of the law of $\delta_\tau x$ at ONE lag (or, $C_\tau$, a second-order quantity): none can detect dependence between scales.
- **Fig B `[CROSS-SCALE]`** — statistics joint across lags, from our draft (ExtremeScattering_7): two-scale coefficients $C_{p,q}(\tau,\tau^*)$ and the time-reversal skewness $S_k(\tau)$.

Run on Jean Zay (needs `saved_results/`).

Bands as in the paper: min/max over 10 batches of trajectories (Fig 5: 30 for the data, 10 for the synthesis), data and synthesis alike. Data dotted, synthesis solid.

| name | label | what |
|---|---|---|
| B | `turbulence_nested_M3_bicw_allmass` | latest scalar potential (constrained windows + all masses), no Scat4 |
| with Scat4 | `turbulence_ggd_mass_final` | previous scalar version (tail-mass fix) + Scat4 |

Units: these runs coarse-grain once (`subseries_len` 512 → `target_len` 256), so one sample = 2 $dt_s$ ≈ 0.21 $\tau_\eta$ and $\tau_\eta$ = 4.67 samples. The smallest lag is ≈ 0.21 $\tau_\eta$ (paper: 0.1), the largest 254 samples ≈ 54 $\tau_\eta$: the $\tau/\tau_\eta = 100$ PDF of Fig 1a cannot be computed and is skipped. The windows are periodized, so the large-$\tau$ end (S₂ peaks near T/2 ≈ 27 $\tau_\eta$) is not comparable to the paper's raw trajectories.

## Assumptions (flagged)
- **Sampling.** One sample of `x1` / the samples = `STEP` raw samples $dt_s$, `STEP = subseries_len / target_len` (= 2 for both runs). The coarse-graining is the Db3 wavelet low-pass of `run_SDE` (`reference_data`), not plain subsampling.
- **$\tau_\eta$ is not measured from our data.** `TAU_ETA` = 9.33 raw samples comes from Li et al. Table 1 ($dt_s$ = 2.25e-3, $\tau_\eta$ = 2.1e-2, same DNS as our file); `TAU_ETA_RUN = TAU_ETA / STEP` = 4.67 samples. Every $\tau/\tau_\eta$ axis and $\tau^*$ depend on it.
- **Units of $\tau$.** Integer lags in samples; plotted as $\tau/\tau_\eta$. Increments are not circular: at lag $\tau$ each trajectory gives $K-\tau$ pairs. Lag $K-1$ is excluded (periodized windows: $x(K-1)=x(0)$, so that increment is 0).
- **Components.** `x1` and the samples are $(n_{traj}, K)$ arrays of ONE velocity component (`load_turbulence_1d`), so item 6 (mixed flatness) is skipped.
- **Normalization.** `x1` is standardized (`normalize`), so $E$, $A$ are not in the paper's physical units. $a$ is the one-sample increment, not divided by $dt$: standardized PDFs and data-vs-synthesis comparisons are unaffected.
- **Periodized windows.** The large-$\tau$ end (S₂ peaks near $K/2$) is not the raw-trajectory one of the paper.
- **$C_{p,q}$.** Both increments start at the same $t$; the numerator and the two marginal moments are averaged over the same $t$-window $0..K-1-\max(\tau,\tau^*)$, so $C_{p,q}=1$ exactly when $|\delta_{\tau^*}x|^p$ and $|\delta_\tau x|^q$ are uncorrelated (grey line). **1 is not the "no cross-scale structure" baseline:** the two increments share $x(t)$ (and overlap in time), so they are correlated even for a Gaussian process. For jointly Gaussian increments with correlation $\rho$, $C_4 = 1+2\rho^2$ (white noise, $\tau^*=1$, $\tau=50$: $\rho=\tfrac12$, $C_4=1.5$, checked numerically). Compare generated with data, not with 1. The diagonal $\tau=\tau^*$ of the $C_4$ map is $F^{(4)}_\tau$ (single-scale).
- **$\tau^*$** = `TAU_STAR_RATIO` $\tau_\eta$, default 2 (inside the crossover range asked for; a parameter, not a measured value).
- **Cross-scale lag grid.** `N_LAGS_CROSS` log-spaced integer lags in $1..K-2$ (the LI panels use every integer lag).
- **Bands.** min/max over `N_BATCHES` disjoint contiguous batches of trajectories; a ratio statistic is evaluated per batch from that batch's averaged moments.
- **Summary table.** max over the lag grid of $|gen-ref|/|ref|$, printed with the lag where it occurs. It is meaningless where $ref\approx 0$ ($C_\tau$ zero crossing, $S_k\approx 0$) or diverges ($\zeta$ near the peak of S₂): read the lag column.

## Existing panels vs the task definitions (reported, NOT changed)
- Items 1, 2, 7 have no batch bands: the paper shows none. Not added.
- Item 1 has no "large" lag: $100\,\tau_\eta$ > the largest lag (≈ 54 $\tau_\eta$), so it is skipped. A reachable value (e.g. 20 or 50) is a choice to make in `TAU_RATIOS`.
- Item 3 also plots $p=8$ (added earlier on request).
- $S_p$ is $\langle(\delta_\tau x)^p\rangle$ with even $p$, identical to $E|\delta_\tau x|^p$.

## Draft inconsistencies (flagged, not silently fixed)
1. **$C_{p,q}$ denominator.** The draft writes $C_{p,q}(\tau,\tau^*) = E[|\delta_{\tau^*}x|^p|\delta_\tau x|^q] / (S_p(\tau)\,S_q(\tau^*))$: the numerator puts $p$ at $\tau^*$, the denominator at $\tau$. Implemented numerator-consistent: $/\,(S_p(\tau^*)\,S_q(\tau))$. The two coincide only for $p=q$ ($C_4$, $C_6$); they differ for $C_{4,2}$.
2. **"Flat4(τ) = log S4 / log S2"** in the draft is not the Li et al. quantity. Only $\zeta(4,\tau)$ of Eq. 4 (ratio of logarithmic derivatives) is implemented (Fig 5).
3. **$C_4$ at $\tau=\tau^*$.** Not assumed maximal there; just plotted ($\tau^*$ marked by a vertical line).
4. **$dE$ is not symmetric in $t$ and $t+\tau$.** Implemented as written, $dE=(x(t+\tau)-x(t))\,x(t)$. Optional variant (clearly marked below): $dE_{sym}=\tfrac12\,(x(t+\tau)^2-x(t)^2)$, odd under time reversal. Algebra: $dE = dE_{sym} - \tfrac12(\delta_\tau x)^2$, so the draft's $dE$ contains a time-even single-lag term and its skewness is not a pure irreversibility measure. That $S_k$ mixes scales (flag `[CROSS-SCALE]`) is our reading, not a result of Li et al.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from run_comparison import load_runs, reference_data
from paper_figures import STYLE, COLOR_1, LS_DATA, LS_SYNTH, _to_np, _apply_style, _finalize
from paper_figures_li import *
from paper_figures_li import _flat, _zeta, _band, _finish

In [ ]:
# ---- parameters (lags in units of tau_eta; converted to samples after loading) ----
N_BATCHES = 10                        # min/max bands over disjoint batches of trajectories (Li et al.)
TAU_RATIOS = (1, 2, 5, 100)           # item 1 lags tau/tau_eta (100 unreachable here, skipped)
SF_ORDERS = (2, 4, 6, 8)              # item 3 (paper: 2, 4, 6; 8 added)
FLAT_ORDERS = (4, 6, 8)               # item 4
ZETA_ORDERS = (4, 6, 8)               # item 5 (paper shows 4)
TAU_STAR_RATIO = 2.0                  # item 8: tau*/tau_eta (crossover range, ASSUMED default)
PQ_PAIRS = ((2, 2), (3, 3), (4, 2))   # item 8: C_4, C_6, C_{4,2}
N_LAGS_CROSS = 40                     # items 8-9: log-spaced integer lags (C_4 map: N x N)

In [ ]:
RUNS = {   # {display name: --label}
    'B: constrained + all masses': 'turbulence_nested_M3_bicw_allmass',
    'with Scat4 (mass_final)': 'turbulence_ggd_mass_final',
}
SEED = 900
SAVE_DIR = None   # e.g. 'figures/li2024' to save pdfs instead of displaying
runs, missing = load_runs(RUNS, seed=SEED)
assert not missing, missing
steps = {k: r['cfg']['subseries_len'] // r['cfg']['target_len'] for k, r in runs.items()}
assert len(set(steps.values())) == 1, steps          # same coarse-graining -> one x1 for all runs
STEP = next(iter(steps.values()))
cfg0 = next(iter(runs.values()))['cfg']
x1 = reference_data(cfg0)
xs = {k: r['xt'] for k, r in runs.items()}
TAU_ETA_RUN, MAX_LAG_RUN = TAU_ETA / STEP, MAX_LAG // STEP
print({k: tuple(v.shape) for k, v in xs.items()}, 'data', tuple(x1.shape))
print(f'step {STEP}: tau_eta = {TAU_ETA_RUN:.2f} samples, max_lag = {MAX_LAG_RUN}')

In [ ]:
# ---- helpers for the cross-scale statistics (x: (n_traj, K) float64, lags in samples) ----
def as_rows(x):
    """(n_traj, K) float64 numpy array from a torch/numpy (n, K) or (n, 1, K) array."""
    x = np.asarray(_to_np(x), dtype=np.float64)
    return x.reshape(x.shape[0], -1)

def increments(x, tau, L=None):
    """delta_tau x(t) = x(t+tau) - x(t) for t = 0 .. L-1 (default: every t), shape (n, L)."""
    L = x.shape[-1] - tau if L is None else L
    return x[:, tau:tau + L] - x[:, :L]

def log_lags(K, n):
    """~n log-spaced integer lags in 1 .. K-2."""
    return np.unique(np.geomspace(1, K - 2, n).round().astype(int))

def ratio_bands(m, f, n_batches=N_BATCHES):
    """m: per-trajectory means, last axis = trajectories; f: the statistic of the averaged moments.
    Returns f on the whole set and its min / max over n_batches disjoint batches of trajectories."""
    fb = np.stack([f(m[..., idx].mean(-1)) for idx in np.array_split(np.arange(m.shape[-1]), n_batches)])
    return f(m.mean(-1)), fb.min(0), fb.max(0)

def two_scale_moments(x, p, q, tau, tau_star):
    """Per-trajectory means of |d*|^p |d|^q, |d*|^p, |d|^q, d* = delta_tau* x(t), d = delta_tau x(t),
    over the SAME t = 0 .. K-1-max(tau, tau*). Shape (3, n_traj)."""
    L = x.shape[-1] - max(tau, tau_star)
    a = np.abs(increments(x, tau_star, L)) ** p
    b = np.abs(increments(x, tau, L)) ** q
    return np.stack([(a * b).mean(1), a.mean(1), b.mean(1)])

def C_pq(M):
    """C_{p,q}(tau, tau*) = E[|d*|^p |d|^q] / (S_p(tau*) S_q(tau)) from two_scale_moments."""
    return M[..., 0] / (M[..., 1] * M[..., 2])

def dE_moments(x, tau, sym=False):
    """Per-trajectory E[dE^k], k = 1, 2, 3, shape (3, n_traj). dE = (x(t+tau) - x(t)) x(t) (draft);
    sym=True: OPTIONAL VARIANT dE_sym = (x(t+tau)^2 - x(t)^2) / 2."""
    L = x.shape[-1] - tau
    dE = 0.5 * (x[:, tau:] ** 2 - x[:, :L] ** 2) if sym else increments(x, tau) * x[:, :L]
    return np.stack([(dE ** k).mean(1) for k in (1, 2, 3)])

def skewness(M):
    """E[(dE - E dE)^3] / E[(dE - E dE)^2]^(3/2) from the raw moments of dE_moments."""
    m1, m2, m3 = M[..., 0], M[..., 1], M[..., 2]
    return (m3 - 3 * m1 * m2 + 2 * m1 ** 3) / (m2 - m1 ** 2) ** 1.5

def plot_vs_lag(lags, ref, gen, name, ylabel, title, fname, logy=False, hline=None, star=True):
    """ref, gen = (statistic, batch min, batch max) on lags; reference dotted, generated solid;
    star: mark tau* with a vertical line."""
    fig, ax = plt.subplots(figsize=STYLE['figsize'])
    t = lags / TAU_ETA_RUN
    for (f, lo, hi), ls, lab in ((ref, LS_DATA, 'data (reference)'), (gen, LS_SYNTH, f'{name} (generated)')):
        _band(ax, t, lo, hi, COLOR_1)
        ax.plot(t, f, ls=ls, color=COLOR_1, lw=STYLE['lw'], label=lab)
    if hline is not None:
        ax.axhline(hline, color='grey', ls=':', lw=2)
    if star:
        ax.axvline(TAU_STAR / TAU_ETA_RUN, color='grey', lw=1)
    ax.set_xscale('log')
    if logy:
        ax.set_yscale('log')
    _apply_style(ax, r'$\tau/\tau_\eta$', ylabel)
    _finish(fig, ax, title, SAVE_DIR, fname, 'pdf', 200)

K = x1.shape[-1]
TAU_STAR = max(1, int(round(TAU_STAR_RATIO * TAU_ETA_RUN)))
LAGS_X = log_lags(K, N_LAGS_CROSS)
sets = {'data': as_rows(x1)} | {name: as_rows(x) for name, x in xs.items()}
print(f'K = {K}; tau* = {TAU_STAR} samples = {TAU_STAR / TAU_ETA_RUN:.2f} tau_eta; '
      f'{len(LAGS_X)} cross-scale lags, {LAGS_X[0]} .. {LAGS_X[-1]} samples')

# Fig A — `[LI]` statistics of Li et al. 2024 (items 1–7, Table 2)
Each is a function of the law of $\delta_\tau x$ at one lag (items 1–5), of the same-time law of two components (item 6), or second order (item 7, equivalent to the spectrum). Scanning them over $\tau$ does not test dependence between scales.

## Item 1 (Fig 1a): standardized PDFs of $\delta_\tau V$ at $\tau/\tau_\eta$ = `TAU_RATIOS` (unreachable lags skipped) — `[LI, single-scale]`

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig1a_increment_pdfs(x1, x, label=name, tau_ratios=TAU_RATIOS, tau_eta=TAU_ETA_RUN,
                                  title='[LI, single-scale]', save_dir=SAVE_DIR)

## Item 2 (Fig 2): acceleration PDF (one-step increment of the coarse-grained signal, 2 $dt_s$) — `[LI, single-scale]`

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig2_acceleration_pdf(x1, x, label=name, title='[LI, single-scale]', save_dir=SAVE_DIR)

## Item 7 (Fig 2 inset): acceleration correlation $C_\tau = \langle a(t+\tau)\,a(t)\rangle$ — `[LI, second order, equivalent to spectrum]`

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig2_inset_acceleration_correlation(x1, x, label=name, max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN,
                                                 title='[LI, second order]', save_dir=SAVE_DIR)

## Item 3 (Fig 4a): structure functions $S_p$, $p$ = `SF_ORDERS` (alpha 1 → 0.4) — `[LI, single-scale]`

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig4a_structure_functions(x1, x, label=name, orders=SF_ORDERS, n_batches=N_BATCHES,
                                       max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN,
                                       title='[LI, single-scale]', save_dir=SAVE_DIR)

## Item 4 (Fig 4b): generalized flatness $F^{(p)}$, $p$ = `FLAT_ORDERS` — `[LI, single-scale]`

In [ ]:
for name, x in xs.items():
    print(name)
    panel_li_fig4b_flatness(x1, x, label=name, orders=FLAT_ORDERS, n_batches=N_BATCHES,
                            max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN,
                            title='[LI, single-scale]', save_dir=SAVE_DIR)

## Item 5 (Fig 5): local scaling exponent $\zeta(p,\tau)$, $p$ = `ZETA_ORDERS` (dotted: non-intermittent $p/2$) — `[LI, single-scale marginals compared across lags]`
Bands: 30 batches for the data, 10 for the synthesis, as the paper's caption.

In [ ]:
for p in ZETA_ORDERS:
    for name, x in xs.items():
        print(name, 'p =', p)
        panel_li_fig5_local_exponent(x1, x, label=name, p=p, max_lag=MAX_LAG_RUN, tau_eta=TAU_ETA_RUN,
                                     title='[LI, single-scale marginals across lags]', save_dir=SAVE_DIR)

## Item 6 (Fig 4c): mixed flatness $F^{(4,ij)}_\tau$ — `[LI, cross-COMPONENT at same τ, not cross-scale]`

In [ ]:
n_comp = 1 if x1.ndim == 2 else x1.shape[1]      # (n_traj, K) = one component; (n_traj, C, K) = C
if n_comp < 2:
    print(f'Item 6 skipped: x1 has shape {tuple(x1.shape)}, one velocity component; '
          'F^(4,ij) needs >= 2 components.')
else:
    raise NotImplementedError(f'{n_comp}-component data: implement F^(4,ij) here')

## Table 2: $E = \langle V^2\rangle$, $A = \langle a^2\rangle$ (x1 is standardized: not the paper's physical units) — `[LI, single-time]`

In [ ]:
for name, x in xs.items():
    print(name)
    li_table2(x1, x)

# Fig B — `[CROSS-SCALE]` statistics joint across lags (items 8–9, from our draft)
## Item 8: two-scale coefficients $C_{p,q}(\tau,\tau^*) = E[|\delta_{\tau^*}x|^p\,|\delta_\tau x|^q] / (S_p(\tau^*)\,S_q(\tau))$ vs $\tau$ at fixed $\tau^*$
$(p,q)$ = `PQ_PAIRS`: $C_4=C_{2,2}$, $C_6=C_{3,3}$, $C_{4,2}$. $\tau^*$ = `TAU_STAR` (vertical line); grey dotted line: 1 (uncorrelated amplitudes; NOT the Gaussian value, see *Assumptions*). Denominator numerator-consistent (see *Draft inconsistencies* 1).

In [ ]:
CPQ = {pq: {k: ratio_bands(np.stack([two_scale_moments(x, *pq, tau, TAU_STAR) for tau in LAGS_X]), C_pq)
            for k, x in sets.items()} for pq in PQ_PAIRS}
for (p, q), res in CPQ.items():
    for name in xs:
        plot_vs_lag(LAGS_X, res['data'], res[name], name, rf'$C_{{{p},{q}}}(\tau,\tau^*)$',
                    '[CROSS-SCALE]', f'cross_C{p}{q}_{RUNS[name]}', logy=True, hline=1)

## Item 8: $C_4(\tau,\tau^*)$ map over (`LAGS_X` × `LAGS_X`) — `[CROSS-SCALE]`
Data, generated, and $\log_{10}$(generated / data). $C_{2,2}$ is symmetric in $(\tau,\tau^*)$; its diagonal is $F^{(4)}_\tau$ (single-scale), the off-diagonal is the cross-scale part. No batch bands on the map.

In [ ]:
def c4_map(x, lags):
    """C_4[i, j] = C_{2,2}(tau = lags[j], tau* = lags[i]); symmetric, so the upper triangle is mirrored."""
    C = np.empty((len(lags), len(lags)))
    for i, ts in enumerate(lags):
        for j in range(i, len(lags)):
            C[i, j] = C[j, i] = C_pq(two_scale_moments(x, 2, 2, lags[j], ts).mean(-1))
    return C

C4 = {k: c4_map(x, LAGS_X) for k, x in sets.items()}
t = LAGS_X / TAU_ETA_RUN
for name in xs:
    Z_d, Z_g = np.log10(C4['data']), np.log10(C4[name])
    Z_r = Z_g - Z_d
    lim = dict(vmin=min(Z_d.min(), Z_g.min()), vmax=max(Z_d.max(), Z_g.max()))
    r = np.abs(Z_r).max()
    fig, axs = plt.subplots(1, 3, figsize=(3 * STYLE['figsize'][0], STYLE['figsize'][1]))
    for ax, Z, ttl, kw in ((axs[0], Z_d, 'data (reference)', lim), (axs[1], Z_g, f'{name} (generated)', lim),
                           (axs[2], Z_r, 'generated / reference', dict(cmap='RdBu_r', vmin=-r, vmax=r))):
        fig.colorbar(ax.pcolormesh(t, t, Z, shading='nearest', **kw), ax=ax)
        ax.set_xscale('log')
        ax.set_yscale('log')
        _apply_style(ax, r'$\tau/\tau_\eta$', r'$\tau^*/\tau_\eta$')
        ax.set_title(rf'[CROSS-SCALE] $\log_{{10}} C_4$: {ttl}', fontsize=STYLE['fontsize'] * 0.5)
    _finalize(fig, SAVE_DIR, f'cross_C4_map_{RUNS[name]}', 'pdf', 200)

## Item 9: time-reversal skewness $S_k(\tau) = E[(dE-E\,dE)^3]/E[(dE-E\,dE)^2]^{3/2}$, $dE = (x(t+\tau)-x(t))\,x(t)$ as in the draft — `[CROSS-SCALE]` (our reading)
Grey dotted line: 0. See *Draft inconsistencies* 4.

In [ ]:
SK = {k: ratio_bands(np.stack([dE_moments(x, tau) for tau in LAGS_X]), skewness) for k, x in sets.items()}
for name in xs:
    plot_vs_lag(LAGS_X, SK['data'], SK[name], name, r'$S_k(\tau)$', '[CROSS-SCALE] (our reading)',
                f'cross_Sk_{RUNS[name]}', hline=0, star=False)

### Item 9 — OPTIONAL VARIANT (not in the draft): $dE_{sym} = \tfrac12\,(x(t+\tau)^2 - x(t)^2)$
Symmetric in $t$, $t+\tau$ and odd under time reversal. Skip this cell to keep only the draft's definition.

In [ ]:
SK_SYM = {k: ratio_bands(np.stack([dE_moments(x, tau, sym=True) for tau in LAGS_X]), skewness)
          for k, x in sets.items()}
for name in xs:
    plot_vs_lag(LAGS_X, SK_SYM['data'], SK_SYM[name], name, r'$S_k(\tau)$, $dE_{sym}$',
                '[CROSS-SCALE] OPTIONAL VARIANT dE_sym', f'cross_Sk_sym_{RUNS[name]}', hline=0, star=False)

# Summary: max relative error generated vs reference over the lag range
$\max_\tau |gen-ref|/|ref|$ and the lag where it occurs (C₄ map: $\tau^*/\tau_\eta$, $\tau/\tau_\eta$). Meaningless where $ref\approx0$ ($C_\tau$ zero crossing, $S_k$) or where $\zeta$ diverges near the peak of S₂: read the lag column. LI items on every integer lag $1..K-2$, cross-scale items on `LAGS_X`.

In [ ]:
def max_rel(ref, gen, *grids):
    """max |gen - ref| / |ref| and the lags (in tau_eta) where it occurs, one per axis of ref."""
    r = np.abs(gen - ref) / np.abs(ref)
    i = np.unravel_index(np.nanargmax(r), r.shape)
    return f'{r[i]:11.3g}  ' + ', '.join(f'{g[k] / TAU_ETA_RUN:.2f}' for g, k in zip(grids, i))

taus_sf, S_ref, _ = structure_functions(x1, N_BATCHES, MAX_LAG_RUN)
taus_a, Ca_ref = acceleration_correlation(x1, MAX_LAG_RUN)
for name, x in xs.items():
    _, S_gen, _ = structure_functions(x, N_BATCHES, MAX_LAG_RUN)
    _, Ca_gen = acceleration_correlation(x, MAX_LAG_RUN)
    rows = [('1 increment PDFs', '[LI, single-scale]', '        n/a  (see figure)'),
            ('2 acceleration PDF', '[LI, single-scale]', '        n/a  (see figure)')]
    rows += [(f'3 S_{p}', '[LI, single-scale]',
              max_rel(S_ref[ORDERS.index(p)], S_gen[ORDERS.index(p)], taus_sf)) for p in SF_ORDERS]
    rows += [(f'4 F_{p}', '[LI, single-scale]', max_rel(_flat(S_ref, p), _flat(S_gen, p), taus_sf))
             for p in FLAT_ORDERS]
    rows += [(f'5 zeta({p})', '[LI, marginals across lags]',
              max_rel(_zeta(S_ref, taus_sf, p), _zeta(S_gen, taus_sf, p), taus_sf)) for p in ZETA_ORDERS]
    rows += [('6 mixed flatness', '[LI, cross-component]', '        n/a  (1 component)'),
             ('7 C_tau', '[LI, second order]', max_rel(Ca_ref[1:], Ca_gen[1:], taus_a[1:]))]
    rows += [(f'8 C_{p},{q}(tau*={TAU_STAR})', '[CROSS-SCALE]', max_rel(res['data'][0], res[name][0], LAGS_X))
             for (p, q), res in CPQ.items()]
    rows += [('8 C_4 map', '[CROSS-SCALE]', max_rel(C4['data'], C4[name], LAGS_X, LAGS_X)),
             ('9 S_k (draft dE)', '[CROSS-SCALE]', max_rel(SK['data'][0], SK[name][0], LAGS_X))]
    if 'SK_SYM' in globals():
        rows += [('9 S_k (dE_sym, optional)', '[CROSS-SCALE]', max_rel(SK_SYM['data'][0], SK_SYM[name][0], LAGS_X))]
    print(f'\n{name}\n{"statistic":26s} {"flag":28s} {"max rel err":>11s}  at tau/tau_eta')
    for stat, flag, val in rows:
        print(f'{stat:26s} {flag:28s} {val}')